# Image Captioning — Beginner-Friendly Tutorial

Welcome! This notebook walks you through **every part** of the Lab 3 image-captioning project, from the big-picture theory all the way down to the individual lines of code. No prior experience with image captioning is required — just a basic familiarity with Python and neural networks.

---

## Table of Contents

1. [What is Image Captioning?](#section-1)
2. [Project Architecture — The Big Picture](#section-2)
3. [Theory: How to Combine Image and Text Embeddings](#section-3)
4. [Project Layout — What Each File Does](#section-4)
5. [Step 1 — Configuration (`config.py`)](#section-5)
6. [Step 2 — Vocabulary (`data/vocabulary.py`)](#section-6)
7. [Step 3 — Data Loading Pipeline](#section-7)
8. [Step 4 — Image Encoder (`models/encoder.py`)](#section-8)
9. [Step 5 — Caption Decoder (`models/decoder.py`)](#section-9)
10. [Step 6 — Training Loop (`training/trainer.py`)](#section-10)
11. [Step 7 — Caption Generation and Evaluation](#section-11)
12. [Putting It All Together — Quick-Start Checklist](#section-12)

---

<a id='section-1'></a>
## 1. What is Image Captioning?

Image captioning is the task of automatically generating a **natural language description** for an image.

```
Input  →  [a JPEG photo of a dog jumping over a fence]
Output →  "A brown dog leaps over a wooden fence."
```

It is an **end-to-end sequence-to-sequence problem**:

| Modality | Representation | Network |
|---|---|---|
| Image | Grid of pixels | Convolutional Neural Network (CNN) |
| Caption | Sequence of words | Recurrent Neural Network (LSTM) |

The CNN turns the image into a **compact feature vector** (a list of numbers that summarise what's in the image). The LSTM then reads that vector and generates a sentence word by word.

### Why is this hard?

- The model must understand **visual content** (objects, scene, relations).
- It must produce **grammatically correct** English.
- Both modalities need to be **fused** sensibly — that's the core design decision explored in this lab.

<a id='section-2'></a>
## 2. Project Architecture — The Big Picture

Here is how the entire system flows from a raw image to a generated caption:

```
┌─────────────┐     ┌──────────────────────────────────────────────────────┐
│  Raw Image  │     │                   ENCODER                            │
│  (JPEG)     │────►│  ResNet-50 backbone → GlobalAvgPool → Linear(256)    │
└─────────────┘     │  Output: image_feature  (batch, 256)                 │
                    └────────────────────────┬─────────────────────────────┘
                                             │  image_feature  (batch, 256)
                                             ▼
                    ┌──────────────────────────────────────────────────────┐
                    │                   DECODER                            │
                    │                                                      │
                    │  At every time step t:                               │
                    │    word_embed = Embedding(w_t)    (batch, 256)       │
                    │    x_t = [image_feature ∥ word_embed]  (batch, 512) │
                    │    h_t, c_t = LSTM(x_t, h_{t-1}, c_{t-1})           │
                    │    logits_t = Linear(h_t)         (batch, vocab)     │
                    │    w_{t+1} = argmax(logits_t)                        │
                    └──────────────────────────────────────────────────────┘
                                             │
                                             ▼
                    "A brown dog leaps over a wooden fence"
```

**Key insight:** The image feature is injected at *every* time step (not just the first). This is the **PAR-inject** strategy — the decoder always has access to the full image context while it decides the next word.

### Training vs. Inference

| Phase | Caption input to LSTM | How the next word is chosen |
|---|---|---|
| **Training** (teacher forcing) | The *ground-truth* previous word | Loss is computed, gradients flow |
| **Inference** (greedy decode) | The *model's own* previous prediction | `argmax` over vocabulary logits |

<a id='section-3'></a>
## 3. Theory: How to Combine Image and Text Embeddings (Task 3.1)

Both the CNN and the word embedding produce a vector of the same dimension (`embed_dim = 256` in this project). How should we fuse them? Five strategies are compared below.

---

### 3.1.1 Concatenation  `[v ∥ w]`

The two vectors are placed side by side to form a longer vector of dimension `2 × embed_dim`.

```python
x = torch.cat([image_feature, word_embed], dim=-1)  # (batch, 512)
```

| Pros | Cons |
|---|---|
| Preserves **all information** from both streams — nothing is discarded | **Doubles** the input dimension, so the LSTM has more parameters |
| The LSTM learns *how* to mix the two streams freely | Slightly higher memory and compute cost |
| Simple to implement | Can slow convergence if dimensions are very large |
| Works well even when the two vectors have very different scales | — |

**This is the method used in this project** (PAR-inject from Tanti & Gatt 2018).

---

### 3.1.2 Addition  `v + w`

The two same-sized vectors are added element-wise.

```python
x = image_feature + word_embed  # (batch, 256)
```

| Pros | Cons |
|---|---|
| **No increase** in dimension — lightweight | Assumes the two spaces are **aligned** — a word embedding and a visual feature may not be |
| Fast and memory-efficient | Information from each source can **cancel** or interfere |
| Fewer parameters than concatenation | Less expressive: same dimension means less capacity to encode both sources |

---

### 3.1.3 Multiplication (Hadamard product)  `v ⊙ w`

Element-wise multiplication — each dimension of one vector gates the corresponding dimension of the other.

```python
x = image_feature * word_embed  # (batch, 256)
```

| Pros | Cons |
|---|---|
| Acts as a **soft gate** — dimensions irrelevant to the image are dampened | If a dimension is near 0 in *either* vector, the product is 0 — **information loss** |
| Captures multiplicative **interactions** between modalities | Sensitive to the magnitude (scale) of each vector |
| No increase in dimension | Hard to initialise well; can produce vanishing values |

---

### 3.1.4 Attention

Instead of using one global image vector, the encoder outputs a **spatial feature map** and the decoder computes a weighted sum over spatial locations at every step.

```python
# Simplified cross-attention
scores  = query @ keys.T       # similarity between current word and each image region
weights = softmax(scores)      # which regions to focus on
context = weights @ values     # weighted image feature
```

| Pros | Cons |
|---|---|
| **Dynamic** — the model looks at different parts of the image for each word | Significantly **more complex** to implement |
| Produces interpretable attention maps (you can visualise what the model is looking at) | Requires the encoder to keep the spatial feature map (larger memory) |
| State-of-the-art performance (used in "Show, Attend and Tell") | More hyperparameters to tune |

---

### 3.1.5 Difference  `v − w`

The signed element-wise difference between the two vectors.

```python
x = image_feature - word_embed  # (batch, 256)
```

| Pros | Cons |
|---|---|
| Captures **mismatches** between image and language — useful in some reasoning tasks | **Asymmetric** — order matters (`v − w ≠ w − v`) |
| Can highlight what is *in the image but not yet described* | Information about shared content is lost (same features cancel) |
| Simple to compute | Rarely used alone; typically combined with other operations |

---

### Summary Table

| Method | Output dim | Preserves info | Complexity | Notes |
|---|---|---|---|---|
| Concatenation | `2d` | ✅ Both | Low | **Used in this project** |
| Addition | `d` | ⚠️ Partial | Very low | Requires aligned spaces |
| Multiplication | `d` | ⚠️ Partial | Very low | Gating effect |
| Attention | `d` | ✅ Spatial | High | Best performance |
| Difference | `d` | ❌ Shared | Very low | Niche use-cases |

<a id='section-4'></a>
## 4. Project Layout — What Each File Does

```
Lab3/
│
├── config.py              ← All hyperparameters and paths in one place
├── main.ipynb             ← The runnable experiment notebook
├── requirements.txt       ← Python packages to install
│
├── data/
│   ├── vocabulary.py      ← Builds the word ↔ index mapping
│   ├── caption_io.py      ← Reads captions.txt and splits images by set
│   ├── caption_dataset.py ← PyTorch Dataset: returns (image, caption) pairs
│   ├── transforms.py      ← Resize / crop / normalise images
│   ├── collate.py         ← Pads captions to the same length inside a batch
│   └── loader.py          ← Orchestrates everything → returns DataLoaders
│
├── models/
│   ├── encoder.py         ← ResNet-50 CNN that produces a feature vector
│   └── decoder.py         ← LSTM that generates captions word by word
│
├── training/
│   └── trainer.py         ← train_epoch() and validate_epoch() functions
│
└── utils/
    ├── evaluation.py      ← generate_caption() and BLEU score calculation
    └── visualization.py   ← Helper to display images with their captions
```

The flow is: **config → data pipeline → models → trainer → evaluation**.

<a id='section-5'></a>
## 5. Step 1 — Configuration (`config.py`)

All hyperparameters live in one file so that you never need to hunt through the code to change a learning rate or image size.

Let's walk through the most important settings:

In [ ]:
import sys, os
# Make sure Python can find our project modules
sys.path.insert(0, os.path.abspath('.'))

import config

print("=== Paths ===")
print(f"  Project root : {config.PROJECT_ROOT}")
print(f"  Images dir   : {config.IMAGES_DIR}")
print(f"  Captions file: {config.CAPTIONS_FILE}")

print("\n=== Device ===")
print(f"  Running on   : {config.DEVICE}")
# If this says 'cuda', training will be ~10-20x faster than on 'cpu'.

print("\n=== Vocabulary ===")
print(f"  Min word freq: {config.VOCAB_MIN_FREQ}")
# Words appearing fewer than 5 times are replaced with <UNK>.

print("\n=== Data Splits ===")
print(f"  Train / Val / Test: {config.TRAIN_SPLIT} / {config.VAL_SPLIT} / {config.TEST_SPLIT}")
# 70% of *images* go to training, 15% to validation, 15% to test.
# Splits are at the image level — the same image never appears in two sets.

print("\n=== Model Architecture ===")
print(f"  embed_dim  : {config.EMBED_DIM}   (word embedding + image projection dimension)")
print(f"  hidden_dim : {config.HIDDEN_DIM}  (LSTM hidden state size)")
print(f"  num_layers : {config.NUM_LAYERS}    (stacked LSTM layers)")
print(f"  dropout    : {config.DROPOUT}  (fraction of neurons randomly zeroed during training)")

print("\n=== Training ===")
print(f"  Batch size  : {config.BATCH_SIZE}")
print(f"  Epochs      : {config.NUM_EPOCHS}")
print(f"  Decoder LR  : {config.DECODER_LR}")
print(f"  Grad clip   : {config.GRAD_CLIP}   (prevents exploding gradients)")

### What is gradient clipping?

LSTMs can suffer from **exploding gradients** — the gradients that flow back through many time steps can grow exponentially large, causing the weights to jump to huge values and the loss to diverge.

Gradient clipping rescales the gradient vector if its total norm exceeds a threshold:

$$\text{if } \|\mathbf{g}\| > \text{clip\_value} \quad\text{then}\quad \mathbf{g} \leftarrow \frac{\text{clip\_value}}{\|\mathbf{g}\|} \cdot \mathbf{g}$$

Setting `GRAD_CLIP = 5.0` means the total gradient norm will never exceed 5.

<a id='section-6'></a>
## 6. Step 2 — Vocabulary (`data/vocabulary.py`)

A neural network can only work with **numbers**, not words. The `Vocabulary` class builds a two-way dictionary:

```
word  →  integer index    (word2idx)
index →  word             (idx2word)
```

### Special tokens

Four reserved tokens are inserted before any real words:

| Token | Index | Meaning |
|---|---|---|
| `<PAD>` | 0 | Padding — fills short captions to match the longest one in a batch |
| `<SOS>` | 1 | Start-of-sequence — the first token fed to the LSTM at inference |
| `<EOS>` | 2 | End-of-sequence — tells the model (and us) the caption is finished |
| `<UNK>` | 3 | Unknown — replaces any word seen fewer than `min_freq` times |

### Example

In [ ]:
from data.vocabulary import Vocabulary

# ── Build a tiny vocabulary from example captions ──────────────────────────
toy_captions = [
    "a dog runs in the park",
    "a cat sits on the mat",
    "a dog and a cat play together in the park",
    "two dogs run fast in the park",
    "a small dog jumps over a fence",
]

vocab = Vocabulary(min_freq=2)   # words must appear at least twice
vocab.build(toy_captions)

print(f"Vocabulary size: {len(vocab)}")
print(f"\nword2idx (first 12): {dict(list(vocab.word2idx.items())[:12])}")

# ── Encode a caption → list of indices ─────────────────────────────────────
sentence = "a dog runs in the park"
encoded = vocab.encode(sentence)
print(f"\nCaption : {sentence}")
print(f"Encoded : {encoded}")
# Note <SOS> at the start and <EOS> at the end

# ── Decode indices → caption ────────────────────────────────────────────────
decoded = vocab.decode(encoded)
print(f"Decoded : {decoded}")

# ── Rare words become <UNK> ─────────────────────────────────────────────────
print(f"\n'jumps' (rare) → index: {vocab['jumps']}  (should be {vocab['<UNK>']}, the <UNK> index)")
print(f"'dog'   (common) → index: {vocab['dog']}")

<a id='section-7'></a>
## 7. Step 3 — Data Loading Pipeline

The data pipeline has five small, focused pieces. Understanding them separately makes the whole easier to follow.

### 7.1 Transforms (`data/transforms.py`)

Raw JPEG images come in all sizes. ResNet-50 expects a **224 × 224** tensor normalised with ImageNet statistics. Two pipelines are defined:

| Pipeline | Used for | Operations |
|---|---|---|
| `_train_transform` | Training set | Resize → **Random**Crop(224) → **RandomHorizontalFlip** → ToTensor → Normalise |
| `_eval_transform` | Val & Test | Resize → **Center**Crop(224) → ToTensor → Normalise |

The random crop and flip are **data augmentation** tricks: they make the model see slightly different views of each image every epoch, which improves generalisation.

Normalisation uses the ImageNet channel means and standard deviations:
$$x_{\text{norm}} = \frac{x - \mu}{\sigma}, \quad \mu = (0.485, 0.456, 0.406),\ \sigma = (0.229, 0.224, 0.225)$$

### 7.2 Dataset (`data/caption_dataset.py`)

Flickr8k has **5 captions per image** (different human annotators described the same image). `CaptionDataset` produces one *(image, caption)* pair per caption — so one image contributes 5 items to the dataset.

### 7.3 Collate (`data/collate.py`)

Captions have **variable lengths**. PyTorch's DataLoader can't stack tensors of different sizes, so we use a custom `collate_captions` function that pads all captions in a batch to the length of the longest one using `<PAD>` tokens.

```
Before padding (batch of 3):       After padding:
  [1, 4, 7, 2]        length 4     [1, 4, 7, 2, 0, 0]
  [1, 5, 8, 9, 2]     length 5     [1, 5, 8, 9, 2, 0]
  [1, 3, 6, 11, 7, 2] length 6     [1, 3, 6, 11, 7, 2]
```

### 7.4 Putting it all together (`data/loader.py`)

The public function `get_data_loaders()` does all of the above in the right order and returns ready-to-use `DataLoader` objects.

In [ ]:
# This cell demonstrates the full data pipeline.
# It requires the Flickr8k dataset to be placed at data/Flickr8k/.
# If you haven't downloaded it yet, read the note below and skip to the next section.

import os
flickr_ready = os.path.isfile(config.CAPTIONS_FILE)

if flickr_ready:
    from data.loader import build_vocabulary_from_train, get_data_loaders, get_split_stats

    # Step 1: load captions and build vocabulary from training data ONLY
    vocab = build_vocabulary_from_train()
    print(f"Vocabulary size: {len(vocab)} words")

    # Step 2: build all DataLoaders
    train_loader, val_loader, test_loader, test_ref_loader, test_refs = \
        get_data_loaders(vocabulary=vocab)

    stats = get_split_stats()
    print(f"\nSplit sizes (images): {stats}")
    print(f"Train batches : {len(train_loader)}")
    print(f"Val   batches : {len(val_loader)}")
    print(f"Test  batches : {len(test_loader)}")

    # Step 3: inspect one batch
    images, captions = next(iter(train_loader))
    print(f"\nOne batch:")
    print(f"  images shape  : {images.shape}   (batch, channels, height, width)")
    print(f"  captions shape: {captions.shape} (batch, padded_seq_len)")
    print(f"  First caption tokens: {captions[0].tolist()}")
    print(f"  Decoded: {vocab.decode(captions[0].tolist())}")
else:
    print("Dataset not found.")
    print("Please download Flickr8k and place it at:")
    print(f"  {config.DATA_DIR}")
    print("\nExpected structure:")
    print("  data/Flickr8k/captions.txt")
    print("  data/Flickr8k/Images/*.jpg")

> **Note — Downloading Flickr8k:**
> 1. Go to https://www.kaggle.com/datasets/adityajn105/flickr8k
> 2. Download and unzip into `data/Flickr8k/`
> 3. Make sure `captions.txt` and an `Images/` folder are present
> 4. Re-run the cell above

<a id='section-8'></a>
## 8. Step 4 — Image Encoder (`models/encoder.py`)

### What is a CNN encoder?

A Convolutional Neural Network (CNN) learns to detect visual patterns — edges, textures, shapes, objects — through layers of learnable filters. We use **ResNet-50**, a well-known architecture that was trained on 1.2 million ImageNet images. We **borrow** its learned visual knowledge (transfer learning) instead of training from scratch.

### Architecture of our encoder

```
Input image  (3, 224, 224)          ← 3-channel RGB, 224×224 pixels
     ↓
ResNet-50 backbone                  ← pretrained, kept frozen by default
  (conv layers only, no classifier)
     ↓  output: (2048, 7, 7)
AdaptiveAvgPool2d(1, 1)             ← squashes the 7×7 grid into a single vector
     ↓  output: (2048,)
Linear(2048 → 256)                  ← learnable projection ← trained from scratch
BatchNorm1d + ReLU
     ↓  output: (256,)              ← image_feature: one vector per image
```

### Freezing vs. fine-tuning

By default `FINE_TUNE_ENCODER = False`: only the projection head trains, the ResNet backbone is frozen. This is faster and avoids overfitting on a small dataset. Setting it to `True` unfreezes the last two ResNet blocks (`layer3`, `layer4`) for end-to-end training.

In [ ]:
import torch
from models.encoder import ImageEncoder

encoder = ImageEncoder(embed_dim=config.EMBED_DIM, fine_tune=config.FINE_TUNE_ENCODER)
encoder = encoder.to(config.DEVICE)

# Count parameters
total   = sum(p.numel() for p in encoder.parameters())
trainable = sum(p.numel() for p in encoder.parameters() if p.requires_grad)
print(f"Total encoder parameters    : {total:,}")
print(f"Trainable encoder parameters: {trainable:,}")
print(f"Frozen (backbone) parameters: {total - trainable:,}")

# Run a dummy forward pass to verify the output shape
dummy_images = torch.randn(4, 3, 224, 224).to(config.DEVICE)  # batch of 4 images
with torch.no_grad():
    features = encoder(dummy_images)

print(f"\nInput  shape: {dummy_images.shape}   (batch=4, channels=3, H=224, W=224)")
print(f"Output shape: {features.shape}         (batch=4, embed_dim={config.EMBED_DIM})")
print("\nEach image is now a single vector of", config.EMBED_DIM, "numbers.")

<a id='section-9'></a>
## 9. Step 5 — Caption Decoder (`models/decoder.py`)

### What is an LSTM?

An LSTM (Long Short-Term Memory) is a type of recurrent neural network that maintains a **hidden state** $h_t$ and a **cell state** $c_t$ that carry information across time steps. The cell state acts as a long-term memory; gates control what to remember and forget.

$$\begin{aligned}
f_t &= \sigma(W_f [h_{t-1}, x_t] + b_f) & \text{forget gate} \\
i_t &= \sigma(W_i [h_{t-1}, x_t] + b_i) & \text{input gate} \\
\tilde{c}_t &= \tanh(W_c [h_{t-1}, x_t] + b_c) & \text{candidate memory} \\
c_t &= f_t \odot c_{t-1} + i_t \odot \tilde{c}_t & \text{new cell state} \\
o_t &= \sigma(W_o [h_{t-1}, x_t] + b_o) & \text{output gate} \\
h_t &= o_t \odot \tanh(c_t) & \text{new hidden state}
\end{aligned}$$

### Decoder architecture

```
At each time step t:

  word_embed = Embedding(w_t)              (batch, 256)
  x_t = concat(image_feature, word_embed)  (batch, 512)  ← concatenation fusion
  h_t, c_t = LSTM(x_t, h_{t-1}, c_{t-1})  (batch, 512)  ← LSTM step
  logits_t = Dropout → Linear(512 → vocab_size)
  next_word = argmax(logits_t)             at inference
```

### Teacher forcing

During **training**, we always feed the *correct* previous word as input (teacher forcing). This makes training stable and fast. During **inference**, we feed the model's *own* previous prediction.

In [ ]:
from models.decoder import CaptionDecoder

# We need a vocab_size — use 5000 as a placeholder if Flickr8k isn't loaded
try:
    vocab_size = len(vocab)
except NameError:
    vocab_size = 5000
    print("(Using placeholder vocab_size = 5000)")

decoder = CaptionDecoder(
    vocab_size  = vocab_size,
    embed_dim   = config.EMBED_DIM,
    hidden_dim  = config.HIDDEN_DIM,
    num_layers  = config.NUM_LAYERS,
    dropout     = config.DROPOUT,
)
decoder = decoder.to(config.DEVICE)

total_dec = sum(p.numel() for p in decoder.parameters())
print(f"Decoder parameters: {total_dec:,}")

# Demonstrate a forward pass with dummy data
batch_size = 4
seq_len    = 10   # 10 time steps (caption length)

dummy_features = torch.randn(batch_size, config.EMBED_DIM).to(config.DEVICE)
dummy_captions = torch.randint(0, vocab_size, (batch_size, seq_len)).to(config.DEVICE)

with torch.no_grad():
    logits = decoder(dummy_features, dummy_captions)

print(f"\nimage_features shape : {dummy_features.shape}  (batch, embed_dim)")
print(f"input captions shape : {dummy_captions.shape}  (batch, seq_len)")
print(f"output logits shape  : {logits.shape}  (batch, seq_len, vocab_size)")
print("\nAt each of the 10 time steps, the model outputs scores for every word in the vocabulary.")
print("The word with the highest score is chosen as the next word.")

<a id='section-10'></a>
## 10. Step 6 — Training Loop (`training/trainer.py`)

### Loss Function: Cross-Entropy

At each time step the model predicts a probability distribution over the vocabulary. Cross-entropy measures how close that distribution is to the ground truth (the correct next word):

$$\mathcal{L} = -\frac{1}{T} \sum_{t=1}^{T} \log P(w_t^* \mid \text{image}, w_1^*, \ldots, w_{t-1}^*)$$

Lower loss = the model is more confident in the correct word.

**PAD tokens are ignored** — they don't contribute to the loss so short captions don't get penalised for padding.

### One Training Step (what happens inside `train_epoch`)

```
for each batch (images, captions):
  1. cap_input  = captions[:, :-1]   # [<SOS>, w1, w2, ..., w_{T-1}]
  2. cap_target = captions[:, 1:]    # [w1,   w2, ..., w_T,   <EOS>]
  3. image_features = encoder(images)
  4. logits = decoder(image_features, cap_input)   # (batch, T, vocab_size)
  5. loss = CrossEntropy(logits, cap_target)        # compare prediction to truth
  6. loss.backward()                               # compute gradients
  7. clip_grad_norm_(parameters, GRAD_CLIP)        # prevent exploding gradients
  8. optimizer.step()                              # update weights
  9. optimizer.zero_grad()                         # clear gradients for next batch
```

### Two Optimisers

The encoder and decoder use **separate optimisers** with different learning rates:

| Component | LR | Reason |
|---|---|---|
| Encoder (projection head) | `1e-4` | Smaller — the backbone is already well-trained |
| Decoder | `4e-4` | Larger — the LSTM is trained from scratch |

If `FINE_TUNE_ENCODER = False`, the encoder optimiser does nothing (there are no trainable backbone parameters).

In [ ]:
import torch.optim as optim
from training.trainer import get_criterion, train_epoch, validate_epoch

# Set up optimisers
criterion     = get_criterion()
enc_optimizer = optim.Adam(encoder.parameters(), lr=config.ENCODER_LR)
dec_optimizer = optim.Adam(decoder.parameters(), lr=config.DECODER_LR)

print("Loss function  :", criterion)
print("Enc optimiser  : Adam, lr =", config.ENCODER_LR)
print("Dec optimiser  : Adam, lr =", config.DECODER_LR)

# ── If the dataset is loaded, run one real training epoch ──────────────────
if 'train_loader' in dir():
    print("\nRunning one training epoch (this may take a few minutes)...")
    train_metrics = train_epoch(
        encoder, decoder, train_loader, criterion,
        enc_optimizer, dec_optimizer, config.DEVICE
    )
    print(f"\nTrain loss: {train_metrics['train_loss']:.4f}")

    val_metrics = validate_epoch(
        encoder, decoder, val_loader, criterion, config.DEVICE
    )
    print(f"Val   loss: {val_metrics['val_loss']:.4f}")
else:
    print("\nSkipping training — load the Flickr8k dataset first.")
    print("Run main.ipynb for the full training loop.")

<a id='section-11'></a>
## 11. Step 7 — Caption Generation and Evaluation

### Greedy Decoding

Once the model is trained, to generate a caption for a new image:

```
1. Encode the image → image_feature
2. Feed <SOS> as the first token
3. For each step:
     logits  = decoder.step(image_feature, current_token, hidden)
     next    = argmax(logits)    ← pick the most likely word
     if next == <EOS>: stop
     append next to caption
4. Join the word list → caption string
```

This is called **greedy decoding** because we always pick the single best word at each step. A more powerful alternative is **beam search**, which tracks multiple hypotheses simultaneously.

### BLEU Score

BLEU (Bilingual Evaluation Understudy) measures how much **n-gram overlap** exists between the generated caption and the reference captions.

$$\text{BLEU-}n = BP \cdot \exp\left(\sum_{k=1}^{n} w_k \log p_k\right)$$

Where:
- $p_k$ = precision of k-grams (fraction of predicted k-grams that appear in references)
- $BP$ = brevity penalty (punishes very short captions)
- $w_k$ = uniform weights

BLEU-4 (4-gram) is the standard benchmark metric for image captioning. A score of ~0.30 is considered reasonable for a simple LSTM model.

In [ ]:
from utils.evaluation import generate_caption
from data.transforms import _eval_transform
from PIL import Image
import matplotlib.pyplot as plt

# ── Only run if Flickr8k is present ────────────────────────────────────────
if 'test_loader' in dir():
    encoder.eval()
    decoder.eval()

    transform = _eval_transform()

    # Grab a few test images and generate captions
    images_dir = config.IMAGES_DIR
    test_images = list(test_refs.keys())[:3]   # first 3 test images

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    for ax, fname in zip(axes, test_images):
        img_path = images_dir / fname
        pil_img  = Image.open(img_path).convert("RGB")
        tensor   = transform(pil_img)

        caption = generate_caption(encoder, decoder, tensor, vocab, config.DEVICE)

        ax.imshow(pil_img)
        ax.set_title(f"Generated:\n{caption}", fontsize=9, wrap=True)
        ax.axis("off")
    plt.tight_layout()
    plt.show()
else:
    print("Dataset not loaded — skipping caption generation demo.")
    print("After loading data and training, generated captions will appear here.")

<a id='section-12'></a>
## 12. Putting It All Together — Quick-Start Checklist

Follow these steps in order to run the full experiment:

### 1. Install dependencies

```bash
pip install -r requirements.txt
```

### 2. Download the Flickr8k dataset

Go to https://www.kaggle.com/datasets/adityajn105/flickr8k and download. Unzip so that the structure is:

```
Lab3/
└── data/
    └── Flickr8k/
        ├── captions.txt
        └── Images/
            ├── 1000268201_693b08cb0e.jpg
            └── ... (8091 images)
```

### 3. Configure your experiment in `config.py`

The defaults are good for a first run. If training is too slow, try:
- `BATCH_SIZE = 64` (if you have enough GPU memory)
- `FINE_TUNE_ENCODER = False` (already the default — keeps training fast)
- Reduce `NUM_EPOCHS` to 5 for a quick sanity check

### 4. Open and run `main.ipynb`

The notebook runs cell-by-cell in this order:

| Cell | What it does |
|---|---|
| Setup | Imports and sets random seeds |
| Data loading | Reads captions, builds vocabulary, creates DataLoaders |
| Model creation | Instantiates encoder and decoder |
| Training loop | Trains for `NUM_EPOCHS`, saves best checkpoint |
| Evaluation | Generates captions on test set, computes BLEU scores |
| Visualisation | Displays sample images with generated captions |

### 5. Check that loss is decreasing

After each epoch, `main.ipynb` prints:
```
Epoch 1/10 — Train Loss: 3.8421 | Val Loss: 3.6150
Epoch 2/10 — Train Loss: 3.1203 | Val Loss: 3.1890
...
```
A decreasing loss means the model is learning. If loss is flat or increasing, check your learning rate and data pipeline.

---

### Common Issues

| Problem | Likely cause | Fix |
|---|---|---|
| `FileNotFoundError` on `captions.txt` | Dataset not downloaded | Follow Step 2 above |
| CUDA out of memory | Batch too large | Reduce `BATCH_SIZE` in `config.py` |
| Loss stays at ~5 | Learning rate too low | Increase `DECODER_LR` to `1e-3` |
| Loss diverges (NaN) | Learning rate too high or no grad clip | Decrease LR; ensure `GRAD_CLIP = 5.0` |
| All captions are identical | Model collapsed | Check vocabulary build; ensure proper shuffling in DataLoader |

---

### Experiment Ideas

Once the baseline works, try these modifications:

1. **Change the embedding fusion method** — edit the decoder to use addition or multiplication instead of concatenation and compare BLEU scores.
2. **Enable fine-tuning** — set `FINE_TUNE_ENCODER = True` and observe whether performance improves.
3. **Increase model capacity** — try `EMBED_DIM = 512`, `HIDDEN_DIM = 1024`.
4. **Add beam search** — instead of greedy argmax, track the top-k hypotheses at each step.